# Week 4: LLM Benchmark with GNN-Generated Insights

Compare Week 3 ML predictions with LLM classifications of GNN-derived network analysis narratives.

**Workflow:**
1. Load GNN-generated text summaries and Week 3 ML predictions
2. Query multiple LLM endpoints to classify the narratives
3. Parse structured outputs (prediction + confidence)
4. Evaluate LLM accuracy vs ML baseline
5. Identify parsing failures and output reliability issues

## Step 1: Imports and Setup

Load libraries for file handling, data processing, API communication, and result parsing.

In [ ]:
# Auto-install and reload dependencies
import subprocess
import sys
import importlib

packages = ['pandas', 'numpy', 'openai']

print("Installing required packages...")
for package in packages:
    try:
        # Try to import first to check if already installed
        __import__(package)
        print(f"  ✓ {package} already installed")
    except ImportError:
        # Not installed, so install it
        print(f"  Installing {package}...")
        result = subprocess.run(
            [sys.executable, '-m', 'pip', 'install', package],
            capture_output=True,
            text=True
        )
        if result.returncode == 0:
            print(f"  ✓ {package} installed successfully")
        else:
            print(f"  ✗ Failed to install {package}")
            print(f"    Error: {result.stderr}")

print("\n✓ All packages ready\n")

# Now import normally
from pathlib import Path
import json
import re

import numpy as np
import pandas as pd

openai = __import__('openai')
OpenAI = openai.OpenAI

print("✓ All imports successful")

## Step 2: Configuration

Set file paths, model endpoints, and output locations.

In [ ]:
ROOT = Path('.').resolve()

# Input files
GNN_INSIGHTS_CSV = ROOT / 'gnn_generated_data' / 'gnn_insights_for_llm.csv'
WEEK3_ML_CSV = ROOT / 'week3_artifacts' / 'ml_predictions.csv'

# Column names
ID_COL = 'sample_id'
TEXT_COL = 'text'
LABEL_COL = 'label'
CONFIDENCE_COL = 'confidence'
ML_PRED_COL = 'prediction'

# LLM endpoints (configure for your Jupyter Hub environment)
MODEL_ENDPOINTS = [
    {'label': 'phi-3.5-mini-instruct', 'model_name': 'Phi-3.5-mini-instruct', 'host': 'localhost', 'port': 8000},
    {'label': 'phi-mini-moe-instruct', 'model_name': 'Phi-mini-MoE-instruct', 'host': 'localhost', 'port': 8001},
    {'label': 'gemma-3-12b-it', 'model_name': 'gemma-3-12b-it', 'host': 'localhost', 'port': 8002},
]
API_KEY = 'EMPTY'
ITERATION_NUMBER = 1

# Output
OUTPUT_DIR = ROOT / 'Results'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RAW_RESULTS_PATH = OUTPUT_DIR / f'gnn_llm_results_raw_{ITERATION_NUMBER}.csv'
CLEAN_RESULTS_PATH = OUTPUT_DIR / f'gnn_llm_results_clean_{ITERATION_NUMBER}.csv'

print(f"GNN Insights CSV: {GNN_INSIGHTS_CSV}")
print(f"Week 3 ML CSV: {WEEK3_ML_CSV}")
print(f"Output directory: {OUTPUT_DIR}")

## Step 3: Load GNN Insights and Week 3 ML Predictions

In [ ]:
if not GNN_INSIGHTS_CSV.exists():
    print(f"ERROR: GNN insights CSV not found at {GNN_INSIGHTS_CSV}")
    print("Please run gnn_insights_to_text.py first to generate the data.")
else:
    gnn_df = pd.read_csv(GNN_INSIGHTS_CSV)
    print(f"Loaded GNN insights: {len(gnn_df)} records")
    display(gnn_df.head())

if WEEK3_ML_CSV.exists():
    ml_df = pd.read_csv(WEEK3_ML_CSV)
    print(f"\nLoaded Week 3 ML predictions: {len(ml_df)} records")
    display(ml_df.head())
    
    # Merge on sample_id
    df = gnn_df[[ID_COL, TEXT_COL, LABEL_COL]].merge(
        ml_df[[ID_COL, ML_PRED_COL]], on=ID_COL, how='left'
    )
    print(f"\nMerged dataset: {len(df)} rows")
else:
    print(f"Warning: Week 3 ML CSV not found at {WEEK3_ML_CSV}")
    print("Proceeding with GNN data only (no ML baseline comparison).")
    df = gnn_df[[ID_COL, TEXT_COL, LABEL_COL]].copy()
    df[ML_PRED_COL] = None

## Step 4: Define Prompt and Parser

Classification task: Interpret GNN network analysis narratives and rate risk level + confidence.

In [ ]:
label_values = sorted(df[LABEL_COL].dropna().astype(str).unique().tolist())
label_set = set(label_values)

print(f"Risk labels found: {label_values}")

def make_prompt(narrative_text, labels):
    """Prompt to interpret GNN network analysis."""
    return (
        'You are a system reliability analyst. '
        'You will read a technical narrative describing a network graph\'s connectivity patterns and infer the underlying system risk.\n\n'
        f'Allowed risk levels: {json.dumps(labels)}\n\n'
        'Output ONLY JSON with schema:\n'
        '{"prediction": "<label>", "confidence": <0_to_1_float>}\n'
        'No extra text.\n\n'
        f'Narrative:\n{narrative_text}'
    )

def parse_response(raw_text, valid_labels):
    """Parse LLM response into structured prediction."""
    output = {'llm_prediction': None, 'llm_confidence': np.nan, 'parse_ok': False, 'parse_error': None}

    if not isinstance(raw_text, str) or not raw_text.strip():
        output['parse_error'] = 'empty_response'
        return output

    m = re.search(r'\{.*\}', raw_text.strip(), flags=re.DOTALL)
    candidate = m.group(0) if m else raw_text.strip()

    try:
        payload = json.loads(candidate)
    except Exception:
        output['parse_error'] = 'invalid_json'
        return output

    pred = str(payload.get('prediction', '')).strip()
    if pred not in valid_labels:
        output['parse_error'] = 'invalid_label'
        return output

    conf = payload.get('confidence', np.nan)
    try:
        conf = float(conf)
    except Exception:
        conf = np.nan

    if not np.isnan(conf) and not (0 <= conf <= 1):
        conf = np.nan

    output['llm_prediction'] = pred
    output['llm_confidence'] = conf
    output['parse_ok'] = True
    return output

## Step 5: LLM Query Functions

Define functions to call endpoints and collect responses.

In [ ]:
SYSTEM_PROMPT = 'You follow output formatting instructions exactly. You are a careful, precise analyst.'

def query_llm(prompt, endpoint_cfg, temperature=0.0, seed=0, timeout=30):
    """Query LLM endpoint with timeout handling."""
    try:
        client = OpenAI(
            api_key=API_KEY,
            base_url=f"http://{endpoint_cfg['host']}:{endpoint_cfg['port']}/v1"
        )
        response = client.chat.completions.create(
            model=endpoint_cfg['model_name'],
            messages=[
                {'role': 'system', 'content': SYSTEM_PROMPT},
                {'role': 'user', 'content': prompt},
            ],
            temperature=temperature,
            seed=seed,
            timeout=timeout
        )
        return response.choices[0].message.content
    except Exception as e:
        return f"ERROR: {str(e)}"

## Step 6: Smoke Test (Single Example)

Test one narrative on each model endpoint before full benchmark.

In [ ]:
example = df.iloc[0]
print(f"Example narrative (Sample {example[ID_COL]}):")
print(f"Text: {example[TEXT_COL][:200]}...")
print(f"Ground truth label: {example[LABEL_COL]}\n")

for endpoint_cfg in MODEL_ENDPOINTS:
    print(f"\nTesting {endpoint_cfg['model_name']} on port {endpoint_cfg['port']}...")
    raw = query_llm(make_prompt(example[TEXT_COL], label_values), endpoint_cfg, temperature=0.0, seed=0)
    parsed = parse_response(raw, label_set)
    print(f"  Raw: {raw[:100]}...")
    print(f"  Parsed: {parsed}")

## Step 7: Full Benchmark Run

Query all models on all narratives. This may take a while if running on many records.

In [ ]:
rows = []
for endpoint_cfg in MODEL_ENDPOINTS:
    print(f"\nRunning benchmark for {endpoint_cfg['model_name']} on port {endpoint_cfg['port']}...")
    for i, row in df.iterrows():
        raw = query_llm(
            make_prompt(str(row[TEXT_COL]), label_values),
            endpoint_cfg,
            temperature=0.0,
            seed=0
        )
        parsed = parse_response(raw, label_set)

        rows.append({
            ID_COL: row[ID_COL],
            'model_label': endpoint_cfg['label'],
            'model_name': endpoint_cfg['model_name'],
            'endpoint_port': endpoint_cfg['port'],
            'prompt_version': 'v1_strict_json',
            'ground_truth': str(row[LABEL_COL]),
            'ml_prediction': str(row[ML_PRED_COL]) if row[ML_PRED_COL] is not None else None,
            'raw_response': raw,
            **parsed
        })

        if (i + 1) % 10 == 0:
            print(f"  Completed {i + 1}/{len(df)}")

results_df = pd.DataFrame(rows)
results_df.to_csv(RAW_RESULTS_PATH, index=False)
print(f'\nSaved raw CSV: {RAW_RESULTS_PATH}')
display(results_df.head(10))

## Step 8: Evaluation and Results Comparison

In [ ]:
eval_df = results_df.copy()
eval_df['llm_prediction_filled'] = eval_df['llm_prediction'].fillna('__INVALID__')

# ML accuracy (if available)
if eval_df['ml_prediction'].notna().any():
    ml_valid = eval_df[eval_df['ml_prediction'].notna()]
    ml_accuracy = float((ml_valid['ground_truth'] == ml_valid['ml_prediction']).mean())
    print(f"Week 3 ML accuracy (on valid records): {ml_accuracy:.3f}")
else:
    ml_accuracy = None
    print("Week 3 ML predictions not available for comparison.")

# LLM summary
llm_summary = eval_df.groupby(['model_label', 'model_name', 'endpoint_port'], as_index=False).agg(
    llm_accuracy=('llm_prediction_filled', lambda s: float((eval_df.loc[s.index, 'ground_truth'] == s).mean())),
    parse_success_rate=('parse_ok', 'mean'),
    invalid_output_rate=('parse_ok', lambda s: float((~s).mean())),
    rows=('model_name', 'size')
)

summary_df = llm_summary.copy()
if ml_accuracy is not None:
    ml_row = pd.DataFrame([{
        'model_label': 'week3_ml',
        'model_name': 'Week 3 ML Baseline',
        'endpoint_port': np.nan,
        'llm_accuracy': ml_accuracy,
        'parse_success_rate': 1.0,
        'invalid_output_rate': 0.0,
        'rows': len(ml_valid)
    }])
    summary_df = pd.concat([ml_row, summary_df], ignore_index=True)

print("\n" + "="*70)
print("RESULTS SUMMARY")
print("="*70)
display(summary_df)

print("\nSample predictions:")
display(eval_df[['model_name', 'ground_truth', 'ml_prediction', 'llm_prediction', 'llm_confidence', 'parse_ok']].head(15))

## Step 9: Save Results and Export Summary

In [ ]:
clean_cols = [
    ID_COL, 'model_label', 'model_name', 'endpoint_port', 'prompt_version', 
    'ground_truth', 'ml_prediction', 'llm_prediction', 'llm_confidence', 'parse_ok', 'parse_error'
]
clean_df = results_df[clean_cols].copy()
clean_df.to_csv(CLEAN_RESULTS_PATH, index=False)

# Per-model exports
for model_label, part in clean_df.groupby('model_label'):
    per_model_clean = OUTPUT_DIR / f'gnn_{model_label}_clean_{ITERATION_NUMBER}.csv'
    per_model_raw = OUTPUT_DIR / f'gnn_{model_label}_raw_{ITERATION_NUMBER}.csv'
    part.to_csv(per_model_clean, index=False)
    results_df[results_df['model_label'] == model_label].to_csv(per_model_raw, index=False)
    print(f'Saved: {per_model_clean}')

print(f'\nMain outputs:')
print(f'  Clean: {CLEAN_RESULTS_PATH}')
print(f'  Raw:   {RAW_RESULTS_PATH}')

# Draft summary for reporting
print("\n" + "="*70)
print("INTERPRETATION SUMMARY")
print("="*70)
print("\nComparison of LLMs vs ML baseline on GNN-derived narratives:")
for _, row in llm_summary.iterrows():
    if ml_accuracy is not None:
        diff = row['llm_accuracy'] - ml_accuracy
        comp = f"(+{diff:.3f})" if diff > 0 else f"({diff:.3f})"
    else:
        comp = "(baseline N/A)"
    print(f"  {row['model_name']}: accuracy={row['llm_accuracy']:.3f} {comp}, parse_success={row['parse_success_rate']:.2%}")